# 🌿 Plant Disease Classifier
Fine-tune a pretrained CNN (EfficientNet / ResNet50) on leaf images to detect 38 crop diseases, then serve it with a Gradio demo.

**Pipeline:** Dataset → Preprocess/Augment → Fine-tune → Evaluate → Deploy → Use

> Run on Google Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

## 1. Setup

In [ ]:
!pip install -q timm gradio scikit-learn

import time, json, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import timm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Config
Switch `MODEL_NAME` to `"resnet50"` to compare against EfficientNet (use `BATCH_SIZE = 32` for ResNet50 on a T4).

In [ ]:
MODEL_NAME  = "efficientnet_b0"
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS_HEAD = 3
EPOCHS_FT   = 7
LR_HEAD     = 1e-3
LR_FT       = 1e-4
WORKERS     = 2
CKPT_PATH   = "best.pt"

## 3. Dataset (PlantVillage, 38 classes)
Uses the Kaggle dataset **New Plant Diseases Dataset (Augmented)** (`vipoooool/new-plant-diseases-dataset`).

1. Create an API token at kaggle.com → Settings → *Create New Token* (downloads `kaggle.json`).
2. Run the cell below and upload `kaggle.json` when prompted.

Already have your own folder of `class_name/*.jpg`? Skip this cell and set `DATA_DIR` in the next one.

In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!pip install -q kaggle
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p data --unzip

In [ ]:
DATA_DIR = None
for p in Path("data").rglob("train"):
    if p.is_dir() and (p.parent / "valid").exists():
        DATA_DIR = p.parent
        break
print("DATA_DIR:", DATA_DIR)

## 4. Preprocess & augment
Training images are randomly cropped, flipped, rotated and color-jittered so the model handles real phone photos. Validation images only get a deterministic resize + center crop.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def pretty(label):
    crop, _, disease = label.partition("___")
    disease = disease.replace("_", " ").strip()
    return f"{crop.replace('_', ' ').strip()} — {disease}" if disease else crop.replace("_", " ")

DATA_DIR = Path(DATA_DIR)
val_dir = next((DATA_DIR / n for n in ("valid", "val") if (DATA_DIR / n).exists()), None)

if (DATA_DIR / "train").exists() and val_dir:
    train_ds = datasets.ImageFolder(DATA_DIR / "train", train_tf)
    val_ds   = datasets.ImageFolder(val_dir, eval_tf)
    classes  = train_ds.classes
else:  # single folder -> 80/20 split
    full_tr = datasets.ImageFolder(DATA_DIR, train_tf)
    full_va = datasets.ImageFolder(DATA_DIR, eval_tf)
    n_val = int(0.2 * len(full_tr))
    g = torch.Generator().manual_seed(SEED)
    tr_idx, va_idx = random_split(range(len(full_tr)), [len(full_tr) - n_val, n_val], generator=g)
    train_ds, val_ds = Subset(full_tr, tr_idx.indices), Subset(full_va, va_idx.indices)
    classes = full_tr.classes

print(f"{len(classes)} classes | {len(train_ds)} train | {len(val_ds)} val")

kw = dict(batch_size=BATCH_SIZE, num_workers=WORKERS, pin_memory=device.type == "cuda")
train_dl = DataLoader(train_ds, shuffle=True, **kw)
val_dl   = DataLoader(val_ds, shuffle=False, **kw)

In [ ]:
x, y = next(iter(train_dl))
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, img, lbl in zip(axes.flat, x, y):
    img = (img.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
    ax.imshow(img); ax.axis("off"); ax.set_title(pretty(classes[lbl])[:22], fontsize=7)
plt.tight_layout(); plt.show()

## 5. Build the model (transfer learning)
Load ImageNet-pretrained weights and swap the final layer for one with our number of classes.

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

def set_backbone_frozen(m, frozen):
    head_ids = {id(p) for p in m.get_classifier().parameters()}
    for p in m.parameters():
        p.requires_grad = (not frozen) or id(p) in head_ids

def run_epoch(loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                out = model(x)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                if scaler:
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (out.argmax(1) == y).sum().item()
            total    += x.size(0)
    return loss_sum / total, correct / total

## 6. Train
**Phase 1** freezes the backbone and trains only the new head (fast, protects pretrained features).
**Phase 2** unfreezes everything and fine-tunes at a 10x lower learning rate.

In [ ]:
history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
best_acc = 0.0

phases = [("Phase 1: head only (backbone frozen)", True,  EPOCHS_HEAD, LR_HEAD),
          ("Phase 2: fine-tune all layers",         False, EPOCHS_FT,   LR_FT)]

for name, frozen, epochs, lr in phases:
    if epochs == 0:
        continue
    set_backbone_frozen(model, frozen)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    print(f"\n=== {name} | trainable params: {sum(p.numel() for p in params):,} ===")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(train_dl, opt)
        vl, va = run_epoch(val_dl)
        sched.step()
        for k, v in zip(history, (ta, va, tl, vl)):
            history[k].append(v)
        print(f"ep {ep}/{epochs} | train {tl:.3f} / {ta:.2%} | val {vl:.3f} / {va:.2%} | {time.time()-t0:.0f}s")
        if va > best_acc:
            best_acc = va
            torch.save({"model": MODEL_NAME, "img_size": IMG_SIZE, "classes": classes,
                        "state_dict": model.state_dict()}, CKPT_PATH)
            print(f"   ✓ saved best ({best_acc:.2%})")

print(f"\nBest validation accuracy: {best_acc:.2%}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 7. Evaluate
Reload the best checkpoint and check per-class performance.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

ck = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ck["state_dict"]); model.eval()

preds, gts = [], []
with torch.no_grad():
    for x, y in tqdm(val_dl):
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        gts   += y.tolist()

names = [pretty(c) for c in classes]
print(classification_report(gts, preds, target_names=names, digits=3))

In [ ]:
cm = confusion_matrix(gts, preds)
fig, ax = plt.subplots(figsize=(13, 13))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(names)), names, rotation=90, fontsize=6)
ax.set_yticks(range(len(names)), names, fontsize=6)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
plt.tight_layout(); plt.show()

# Most-confused pairs
off = cm.copy(); np.fill_diagonal(off, 0)
print("Top confusions:")
for idx in np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:5], off.shape))[0]:
    t, p = idx
    if off[t, p]:
        print(f"  {names[t]}  ->  {names[p]}: {off[t, p]}")

## 8. Grad-CAM: see *where* the model looks
A heatmap of the image regions that drove the prediction. A good model should focus on lesions, not the background.

In [ ]:
import torch.nn.functional as F

def find_last_conv(m):
    last = None
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            last = mod
    return last

def gradcam(pil_img):
    x = eval_tf(pil_img.convert("RGB")).unsqueeze(0).to(device)
    feats, grads = [], []
    layer = find_last_conv(model)
    h1 = layer.register_forward_hook(lambda m, i, o: feats.append(o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.append(go[0]))
    model.zero_grad()
    out = model(x)
    cls = out.argmax(1).item()
    out[0, cls].backward()
    h1.remove(); h2.remove()
    w = grads[0].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats[0]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), cls, torch.softmax(out, 1)[0, cls].item()

raw_val = datasets.ImageFolder(val_dir if val_dir else DATA_DIR)
sample = random.sample(range(len(raw_val)), 4)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j, i in enumerate(sample):
    img, _ = raw_val[i]
    cam, cls, conf = gradcam(img)
    disp = img.resize((IMG_SIZE, IMG_SIZE))
    axes[0, j].imshow(disp); axes[0, j].axis("off")
    axes[1, j].imshow(disp); axes[1, j].imshow(cam, cmap="jet", alpha=0.45); axes[1, j].axis("off")
    axes[0, j].set_title(f"{pretty(classes[cls])[:24]}\n{conf:.0%}", fontsize=8)
plt.tight_layout(); plt.show()

## 9. Deploy: Gradio demo
Upload a leaf photo and get the top-5 predicted diseases with confidence. `share=True` gives a public link.

In [ ]:
import gradio as gr

model.eval()

def predict(img):
    if img is None:
        return {}
    x = eval_tf(img.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    top = torch.topk(probs, k=min(5, len(classes)))
    return {pretty(classes[i]): float(p) for p, i in zip(top.values, top.indices)}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload a leaf photo"),
    outputs=gr.Label(num_top_classes=5, label="Prediction"),
    title="🌿 Plant Disease Classifier",
    description="Fine-tuned CNN that detects crop diseases from a single leaf image. "
                "Best with a close-up of one leaf on a plain background.",
)
demo.launch(share=True, debug=False)

## 10. Save your model
Download `best.pt`, or copy it to Google Drive. To host the demo for free, put it in a Hugging Face Space with an `app.py` like the cell above.

In [ ]:
from google.colab import files
files.download(CKPT_PATH)

### Notes
- PlantVillage images are lab-style (single leaf, plain background), so accuracy on messy field photos will be lower. Add field images (e.g. PlantDoc) or Indian crop data and retrain to improve.
- Ideas to extend: treatment recommendations per disease, ONNX/TFLite export for a mobile app, model comparison (ResNet50 vs EfficientNet) in a results table.